# 02c — Preparación de modelado

Bloques 6–8 del cuaderno 02: MLP (aparcado), PCA de CLC y normalización.

## Imports (celda única del cuaderno)

In [ ]:
# Imports del cuaderno (celda única) — todas las dependencias aquí; las demás celdas no importan nada.
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from wildfire.data.io import load_csv, save_csv


# 6 — One-hot encoding y red neuronal

Este notebook usa nuestro para preparar variables categóricas con one-hot encoding y entrenar una red neuronal de clasificación mediante `MLPClassifier` de scikit-learn. Por defecto, predice la columna `confidence_cat`, que contiene la clase de la detección. Puedes cambiar `TARGET` en la primera celda si tu objetivo es otra variable categórica.

Se separan los datos en entrenamiento y prueba antes de ajustar el preprocesamiento. Las variables categóricas se imputan y codifican; las numéricas se imputan y estandarizan. La métrica en el conjunto de prueba ofrece una primera evaluación, no una validación operacional ni causal.

**Bloque aparcado:** se conserva como andamiaje del pipeline (imputación → one-hot/escalado → clasificador), no como resultado. El objetivo es la confianza reportada por el sensor con predictores de datos cotidianos: sus métricas validan que el pipeline corre, no son hallazgos sobre incendios. La vía de modelado es ocurrencia (`processing/occurrence.py` + días de fondo futuros); las columnas de satélite están excluidas.

### 6.1 — Carga y objetivo

Carga el CSV y fija `TARGET`; muestra su distribución.

In [ ]:

df = load_csv("data/processed/expanded/firms_spain_final_lags_index.csv")
TARGET = 'confidence_cat'  # Cambia este valor si quieres predecir otra variable categórica.
if TARGET not in df.columns:
    raise ValueError(f'No existe la variable objetivo {TARGET!r}.')
print(f'Filas: {len(df):,}; columnas: {df.shape[1]}')
print('Distribución de la variable objetivo:')
display(df[TARGET].value_counts(dropna=False).rename_axis(TARGET).to_frame('n'))


## Preparar variables

Se excluyen `acq_date` y `acq_time` como texto/códigos sin transformar; para conservar parte de la fecha se derivan mes y día del año. Los campos CLC y otros códigos nominales se tratan como categorías, aunque estén guardados como números.

### 6.2 — Predictores X/y

Deriva mes/día y separa numéricas y categóricas.

In [ ]:
data = df.dropna(subset=[TARGET]).copy()

if 'acq_date' in data.columns:
    dates = pd.to_datetime(data['acq_date'], errors='coerce')
    data['acq_month'] = dates.dt.month
    data['acq_day_of_year'] = dates.dt.dayofyear

drop_cols = [c for c in ['acq_date', 'acq_time'] if c in data.columns]
X = data.drop(columns=[TARGET] + drop_cols)
y = data[TARGET].astype(str)

# Códigos CLC y etiquetas/códigos que representan categorías, no magnitudes.
explicit_categorical = {c for c in X.columns if c.lower().startswith('clc_')}
explicit_categorical.update(c for c in ['satellite', 'instrument', 'daynight', 'country', 'ccaa', 'confidence_cat', 'confidence_og_cat'] if c in X.columns)
explicit_categorical.update(c for c in X.columns if pd.api.types.is_bool_dtype(X[c]))
categorical_cols = sorted(set(X.select_dtypes(include=['object', 'category']).columns) | explicit_categorical)
numeric_cols = [c for c in X.select_dtypes(include=[np.number]).columns if c not in categorical_cols]

if not numeric_cols and not categorical_cols:
    raise ValueError('No hay variables predictoras disponibles después de excluir el objetivo.')
print(f'Predictores numéricos ({len(numeric_cols)}): {numeric_cols}')
print(f'Predictores categóricos ({len(categorical_cols)}): {categorical_cols}')
display(pd.DataFrame({'tipo': ['numérica'] * len(numeric_cols) + ['categórica'] * len(categorical_cols),
                      'columna': numeric_cols + categorical_cols}))

## Separación, one-hot encoding y entrenamiento

Las etiquetas de clase se codifican como enteros para el clasificador. El one-hot encoding de los predictores categóricos ocurre dentro del pipeline y se ajusta únicamente con el conjunto de entrenamiento, lo que evita que el preprocesamiento aprenda categorías desde el conjunto de prueba.

### 6.3 — Partición y entrenamiento

Split estratificado, ColumnTransformer y MLP con early stopping.

In [ ]:
class_counts = y.value_counts()
if class_counts.min() < 2:
    raise ValueError('Cada clase necesita al menos 2 filas para crear una partición estratificada. Revisa la distribución del objetivo.')

X_train, X_test, y_train_text, y_test_text = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_text)
y_test = label_encoder.transform(y_test_text)

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
try:
    one_hot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:  # compatibilidad con versiones anteriores de scikit-learn
    one_hot = OneHotEncoder(handle_unknown='ignore', sparse=False)
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', one_hot)
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
], remainder='drop')

model = Pipeline([
    ('preprocessor', preprocessor),
    ('neural_network', MLPClassifier(
        hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
        alpha=0.0001, batch_size='auto', learning_rate_init=0.001,
        max_iter=100, early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=10, random_state=42
    ))
])
model.fit(X_train, y_train)
print('Forma después del preprocesamiento:', model.named_steps['preprocessor'].transform(X_train).shape)
print('Iteraciones de entrenamiento:', model.named_steps['neural_network'].n_iter_)
print('¿Convergió según scikit-learn?:', not model.named_steps['neural_network']._no_improvement_count > 10)

### 6.4 — Métricas y matriz de confusión

Accuracy, balanced accuracy, informe por clase y matriz.

In [ ]:
y_pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print(f'Balanced accuracy: {balanced_accuracy_score(y_test, y_pred):.3f}')
print('\nInforme por clase:')
print(classification_report(y_test, y_pred, labels=np.arange(len(label_encoder.classes_)), target_names=label_encoder.classes_, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    labels=np.arange(len(label_encoder.classes_)),
    display_labels=label_encoder.classes_,
    cmap='Blues', xticks_rotation=45, values_format='d'
)
plt.title('Matriz de confusión — conjunto de prueba')
plt.tight_layout()
plt.show()

# 7 — PCA para explorar la reducción de columnas CLC

Este análisis localiza todas las columnas cuyo nombre comienza por `clc_`. Como las clases CLC son códigos categóricos, primero se aplica codificación one-hot; introducir los códigos directamente como números haría que PCA interpretara distancias ordinales que no necesariamente existen. A continuación se aplica PCA a esas categorías codificadas.

El notebook informa cuántos componentes hacen falta para conservar el 90% y el 95% de la varianza de las columnas CLC, muestra la varianza acumulada y permite inspeccionar qué columnas originales contribuyen más a los componentes. Si el CSV supera 20.000 filas, se ajusta PCA sobre una muestra reproducible de 20.000 filas para mantener el análisis manejable. El resultado es exploratorio; la decisión final depende del modelo y de la interpretación que se quiera conservar.

### 7.1 — Detección de columnas CLC

Localiza las `clc_*`; informa tipos, distintos y faltantes.

In [ ]:

df = load_csv("data/processed/expanded/firms_spain_final_lags_index.csv")
clc_cols = [col for col in df.columns if col.lower().startswith('clc_')]
if not clc_cols:
    raise ValueError('No se encontraron columnas cuyo nombre empiece por clc_.')

print(f'Filas en el CSV: {len(df):,}')
print(f'Columnas CLC detectadas ({len(clc_cols)}): {clc_cols}')
display(pd.DataFrame({
    'tipo': df[clc_cols].dtypes.astype(str),
    'valores_distintos': df[clc_cols].nunique(dropna=True),
    'faltantes': df[clc_cols].isna().sum()
}))


## Codificación y ajuste de PCA

Los faltantes se conservan como una categoría explícita (`SIN_DATO`). El codificador se ajusta con todas las filas para conocer las categorías presentes; PCA se ajusta con una muestra de hasta 20.000 filas. One-hot produce una matriz dispersa primero y se convierte a matriz densa solo para la muestra usada por PCA.

### 7.2 — One-hot y ajuste

Clases a texto (+SIN_DATO); PCA en muestra de 20.000.

In [ ]:
MAX_FILAS_PCA = 20_000

# Convertir todas las clases a texto previene que etiquetas mixtas causen errores.
X_cat = df[clc_cols].astype('string').fillna('SIN_DATO').astype(str)
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
except TypeError:  # compatibilidad con versiones anteriores de scikit-learn
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=True)

encoder.fit(X_cat)
n_fit = min(len(X_cat), MAX_FILAS_PCA)
X_sample_cat = X_cat.sample(n=n_fit, random_state=42)
X_sample_sparse = encoder.transform(X_sample_cat)
X_sample = X_sample_sparse.toarray()

n_components = min(X_sample.shape[0], X_sample.shape[1])
pca = PCA(n_components=n_components, svd_solver='full', random_state=42)
scores = pca.fit_transform(X_sample)
explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

def components_for(threshold):
    reached = np.flatnonzero(cumulative >= threshold)
    return int(reached[0] + 1) if len(reached) else None

n90, n95 = components_for(0.90), components_for(0.95)
n_encoded = X_sample.shape[1]
print(f'Columnas CLC originales: {len(clc_cols)}')
print(f'Variables one-hot: {n_encoded}')
print(f'Filas usadas para ajustar PCA: {n_fit:,}')
print(f'Componentes para >=90% de varianza: {n90}')
print(f'Componentes para >=95% de varianza: {n95}')
if n95 is not None:
    print(f'Reducción de {n_encoded} variables one-hot a {n95} componentes para conservar >=95% de varianza.')

variance_table = pd.DataFrame({
    'componente': np.arange(1, len(explained) + 1),
    'varianza_individual': explained,
    'varianza_acumulada': cumulative
})
display(variance_table.head(20))

### 7.3 — Varianza explicada

Scree y acumulada con umbrales 90%/95%.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(variance_table['componente'], variance_table['varianza_individual'], marker='o', ms=3)
axes[0].set(title='Varianza explicada por componente', xlabel='Componente principal', ylabel='Proporción de varianza')
axes[1].plot(variance_table['componente'], variance_table['varianza_acumulada'], marker='o', ms=3)
axes[1].axhline(.90, color='orange', linestyle='--', label='90%')
axes[1].axhline(.95, color='red', linestyle='--', label='95%')
axes[1].set(title='Varianza explicada acumulada', xlabel='Número de componentes', ylabel='Proporción acumulada', ylim=(0, 1.02))
axes[1].legend()
plt.tight_layout()
plt.show()

## Contribuciones de las columnas CLC

Para cada componente se suman las magnitudes absolutas de las cargas de sus categorías one-hot, agrupadas por la columna CLC original. Es una ayuda para interpretar qué campos aportan más a cada componente; las cargas individuales dependen de la categoría concreta. La última celda deja en memoria las puntuaciones PCA de la muestra, reducidas al menor número de componentes que conserva el 95% (o todos si no se alcanza ese umbral).

### 7.4 — Contribuciones y muestra reducida

Suma de |cargas| por columna CLC; `reduced_sample` solo en memoria.

In [ ]:
encoded_names = encoder.get_feature_names_out(clc_cols)
# El orden de get_feature_names_out sigue categorías_ por cada columna.
encoded_to_source = []
for col, categories in zip(clc_cols, encoder.categories_):
    encoded_to_source.extend([col] * len(categories))

n_show = min(5, pca.components_.shape[0])
loading_rows = []
for pc in range(n_show):
    abs_loadings = np.abs(pca.components_[pc])
    summary = pd.Series(abs_loadings).groupby(encoded_to_source).sum().sort_values(ascending=False)
    for source_col, value in summary.head(5).items():
        loading_rows.append({'componente': f'PC{pc+1}', 'columna_CLC': source_col, 'suma_cargas_absolutas': value})
display(pd.DataFrame(loading_rows))

# DataFrame reducido de la muestra usada por PCA.
n_keep = n95 if n95 is not None else scores.shape[1]
reduced_sample = pd.DataFrame(
    scores[:, :n_keep],
    index=X_sample_cat.index,
    columns=[f'PC{i+1}' for i in range(n_keep)]
)
print(f'Muestra reducida: {reduced_sample.shape[0]:,} filas x {reduced_sample.shape[1]} componentes')
reduced_sample.head()

# 8 — Variables normalizadas

Este notebook carga nuestro dataframe y normaliza las variables numéricas continuas con **Z-score** (`StandardScaler`): centra cada variable alrededor de 0 y la escala para que su desviación típica sea aproximadamente 1.

Se excluyen identificadores y códigos numéricos categóricos como `type`, `sensor`, `acq_time` y las columnas `clc_`; normalizar esos códigos les daría un significado de magnitud que no tienen. Las variables categóricas y los valores faltantes se conservan. El CSV de salida es una copia donde solo se sustituyen las columnas seleccionadas.

La celda de exportación ajusta el escalador al conjunto completo porque genera una versión normalizada descriptiva del archivo. Si se usa para evaluar un modelo predictivo, hay que dividir primero entrenamiento/prueba y ajustar el escalador solo con entrenamiento, como muestra la última sección.

### 8.1 — Carga

Carga el CSV con índice; informa filas×columnas.

In [ ]:

df = load_csv("data/processed/expanded/firms_spain_final_lags_index.csv")
print(f'{len(df):,} filas y {df.shape[1]} columnas')
df.head()


## Seleccionar variables

Además de las columnas CLC, se excluyen columnas que son códigos o etiquetas, aunque estén almacenadas como números. Las variables booleanas también quedan fuera. La lista de exclusión se puede ajustar según el uso posterior.

### 8.2 — Selección de variables

Numéricas continuas excluyendo códigos CLC y booleanos.

In [ ]:
exclude = {'type', 'sensor', 'acq_time', 'is_day'}
exclude.update(c for c in df.columns if c.lower().startswith('clc_'))
numeric_cols = [
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in exclude and not pd.api.types.is_bool_dtype(df[c])
]
if not numeric_cols:
    raise ValueError('No se encontraron variables numéricas continuas para normalizar.')
print(f'Variables numéricas seleccionadas ({len(numeric_cols)}):')
print(numeric_cols)
display(pd.DataFrame({
    'faltantes': df[numeric_cols].isna().sum(),
    'valores_distintos': df[numeric_cols].nunique(dropna=True),
    'media_original': df[numeric_cols].mean(),
    'desviacion_original': df[numeric_cols].std()
}).sort_index())

## Aplicar Z-score y guardar copia

`StandardScaler` conserva los NaN. No imputa faltantes: si el siguiente modelo los requiere, habrá que imputarlos dentro del pipeline de entrenamiento.

### 8.3 — Z-score y guardado

Estandariza las seleccionadas (conserva NaN) y guarda la copia.

In [ ]:
scaler = StandardScaler()
df_norm = df.copy()
df_norm[numeric_cols] = scaler.fit_transform(df[numeric_cols])

comparison = pd.DataFrame({
    'media_normalizada': df_norm[numeric_cols].mean(),
    'desviacion_normalizada': df_norm[numeric_cols].std(),
    'faltantes': df_norm[numeric_cols].isna().sum()
})
display(comparison)

output_path = 'data/processed/expanded/firms_spain_final_normalized.csv'
save_csv(df_norm, output_path)
print(f'Archivo normalizado guardado en: {output_path}')
df_norm.head()

## Ejemplo correcto para evaluación predictiva

Al evaluar un modelo, ajustar el escalador con todo el CSV filtraría información del conjunto de prueba. En este ejemplo se divide primero y se aprende la media/desviación únicamente en entrenamiento; después se transforma prueba con esos mismos parámetros. Para una red neuronal, lo ideal es integrar este escalador en un `Pipeline`.

### 8.4 — Ejemplo train-only

Escalador ajustado solo en entrenamiento; patrón para evaluación.

In [ ]:
X = df[numeric_cols]
X_train, X_test = train_test_split(X, test_size=0.20, random_state=42)
scaler_train = StandardScaler()
X_train_norm = pd.DataFrame(scaler_train.fit_transform(X_train), columns=numeric_cols, index=X_train.index)
X_test_norm = pd.DataFrame(scaler_train.transform(X_test), columns=numeric_cols, index=X_test.index)
print('Filas entrenamiento:', len(X_train_norm), '| filas prueba:', len(X_test_norm))
display(X_train_norm.head())